In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.
Path to dataset files: /kaggle/input/gtsrb-german-traffic-sign


In [2]:
import os
print(os.listdir(path))

['Meta', 'meta', 'Meta.csv', 'Train.csv', 'Test.csv', 'Test', 'test', 'Train', 'train']


In [3]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/gtsrb-german-traffic-sign/Train.csv")
test_df = pd.read_csv("/kaggle/input/gtsrb-german-traffic-sign/Test.csv")

print(train_df.head())
print(train_df.shape)

   Width  Height  Roi.X1  Roi.Y1  Roi.X2  Roi.Y2  ClassId  \
0     27      26       5       5      22      20       20   
1     28      27       5       6      23      22       20   
2     29      26       6       5      24      21       20   
3     28      27       5       6      23      22       20   
4     28      26       5       5      23      21       20   

                             Path  
0  Train/20/00020_00000_00000.png  
1  Train/20/00020_00000_00001.png  
2  Train/20/00020_00000_00002.png  
3  Train/20/00020_00000_00003.png  
4  Train/20/00020_00000_00004.png  
(39209, 8)


In [4]:
print(train_df['ClassId'].unique())
print("Total Classes:", train_df['ClassId'].nunique())

[20  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42]
Total Classes: 43


In [5]:
meta = pd.read_csv("/kaggle/input/gtsrb-german-traffic-sign/Meta.csv")

print(meta[['ClassId']].head(50))

    ClassId
0        27
1         0
2         1
3        10
4        11
5        12
6        13
7        14
8        15
9        16
10       17
11       18
12       19
13        2
14       20
15       21
16       22
17       23
18       24
19       25
20       26
21       28
22       29
23        3
24       30
25       31
26       32
27       33
28       34
29       35
30       36
31       37
32       38
33       39
34        4
35       40
36       41
37       42
38        5
39        6
40        7
41        8
42        9


In [6]:
selected_classes = [14,17,0,1,2,3,4,5]

train_df = train_df[
    train_df['ClassId'].isin(selected_classes)
]

test_df = test_df[
    test_df['ClassId'].isin(selected_classes)
]

In [14]:
import cv2
import numpy as np
import os
from tqdm import tqdm

IMG_SIZE = 64
DATASET_DIR = "/kaggle/input/gtsrb-german-traffic-sign"

images = []
labels = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):

    img_path = os.path.join(DATASET_DIR, row['Path'])

    img = cv2.imread(img_path)

    if img is None:
        print(f"Could not read: {img_path}")
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(row['ClassId'])

X = np.array(images, dtype=np.float32) / 255.0
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

100%|██████████| 11820/11820 [01:11<00:00, 164.41it/s]


X shape: (11820, 64, 64, 3)
y shape: (11820,)


In [16]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()

y = le.fit_transform(y)

y = to_categorical(y)

In [17]:
from sklearn.model_selection import train_test_split

X_train,X_val,y_train,y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train)

CNN

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

num_classes = y.shape[1]

model = Sequential([

    Conv2D(32,(3,3),activation='relu',
           input_shape=(64,64,3)),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(),

    Flatten(),

    Dense(256,activation='relu'),
    Dropout(0.5),

    Dense(num_classes,
          activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,275,208 (4.86 MB)

 Trainable params: 1,275,208 (4.86 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [21]:
history = model.fit(
    datagen.flow(
        X_train,
        y_train,
        batch_size=32
    ),
    validation_data=(X_val,y_val),
    epochs=15
)

Epoch 1/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 23s 57ms/step - accuracy: 0.4193 - loss: 1.3920 - val_accuracy: 0.5355 - val_loss: 1.0850
Epoch 2/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.5684 - loss: 1.0591 - val_accuracy: 0.8135 - val_loss: 0.6338
Epoch 3/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.7378 - loss: 0.6961 - val_accuracy: 0.9255 - val_loss: 0.2673
Epoch 4/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - accuracy: 0.8385 - loss: 0.4514 - val_accuracy: 0.9619 - val_loss: 0.1405
Epoch 5/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.8983 - loss: 0.2931 - val_accuracy: 0.9776 - val_loss: 0.0896
Epoch 6/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 20s 43ms/step - accuracy: 0.9253 - loss: 0.2195 - val_accuracy: 0.9805 - val_loss: 0.0588
Epoch 7/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.9453 - loss: 0.1668 - val_accuracy: 0.9882 - val_loss: 0.0500
Epoch 8/15
296/296 ━━━━━━━━━━━━━━━━━━━━ 13s 44ms/step - accuracy: 0.9596 - loss: 0.1265 - 

In [22]:
loss,acc = model.evaluate(
    X_val,
    y_val
)

print("Accuracy:",acc)

74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9979 - loss: 0.0060
Accuracy: 0.9978849291801453


In [23]:
model.save("traffic_sign_model.h5")

In [24]:
from google.colab import files

files.download("traffic_sign_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>